In [1]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv("ebay_cleaned_english_auto24_format.csv", delimiter=',' ,encoding='latin-1')

data.head()

,make,model,price_eur,log_price,mileage_km,power_ps,power_kw,car_age,fuel_type,transmission,body_type,damaged
0,jeep,grand cherokee,9800,9.190240,125000,163,119.9,22,diesel,automatic,suv,NaN
1,volkswagen,golf,1500,7.313887,150000,75,55.2,25,petrol,manual,hatchback,False
2,skoda,fabia,3600,8.188967,90000,69,50.7,18,diesel,manual,hatchback,False
3,bmw,316i,650,6.478510,150000,102,75.0,31,petrol,manual,sedan,True
4,peugeot,206,2200,7.696667,150000,109,80.2,22,petrol,manual,cabrio,False


In [11]:
#Splitting the data

from sklearn.model_selection import train_test_split

numeric_features = ['power_ps', 'mileage_km', 'car_age']
categorical_features = ['body_type', 'transmission', 'model', 'fuel_type', 'make', 'damaged', ]
#data[categorical_features] = data[categorical_features].fillna("unknown")

y = data['log_price'].copy()
X = data[categorical_features + numeric_features].copy()

X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, random_state = 42, test_size = 0.2)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, random_state = 42, test_size = 0.2)





In [9]:
#Standard Random Forest Regressor
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import ParameterGrid

ohe = OneHotEncoder(handle_unknown = "ignore", sparse_output = False, max_categories = 50)

seed = 42

def make_forest(criterion = "squared_error", max_depth = 3, min_samples_leaf = 5):
    prepare = ColumnTransformer([
        ("numeric", SimpleImputer(strategy = "mean"), numeric_features),
        ("categorical", ohe, categorical_features)
    ])
    return Pipeline([
        ("prepare", prepare),
        ("forest", RandomForestRegressor(criterion=criterion, max_depth=max_depth, min_samples_leaf=min_samples_leaf, n_jobs=-1, random_state = seed)),
         ])

#Defining the hyperparameters and running the model:

def holdout_table(parameter_choices):
    rows, models = [], []
    for i, settings in enumerate(ParameterGrid(parameter_choices), start=1):
        t0 = time.time()
        model = make_forest(**settings).fit(X_train, y_train)
        models.append(model)
        probs_train = model.predict(X_train)
        probs_val = model.predict(X_val)
        
        rows.append({
            "settings":settings,
            "training MAE": mean_absolute_error(y_train, probs_train), #which measure to use?!
            "validation MAE": mean_absolute_error(y_val, probs_val), #which measure to use?!
            "training RMSE": root_mean_squared_error(y_train, probs_train),
            "validation RMSE": root_mean_squared_error(y_val, probs_val),
            "val_r2": r2_score(y_val, probs_val),
            "avg_leaves": sum(t.get_n_leaves() for t in model.named_steps["forest"].estimators_) / len(model.named_steps["forest"].estimators_),
        })
        print(f"[{i}/9] Finished {settings} in {time.time() - t0:.1f}s")
    return pd.DataFrame(rows), models

depth_results, depth_models = holdout_table({
    "criterion" : ["squared_error"], "max_depth" : [3, 6, 9], "min_samples_leaf" : [5, 10, 20]
})
depth_labels = [str(settings["max_depth"]) for settings in depth_results["settings"]]

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(depth_results.round(3))




[1/9] Finished {'criterion': 'squared_error', 'max_depth': 3, 'min_samples_leaf': 5} in 8.1s


KeyboardInterrupt: 

In [30]:
auto24 = pd.read_csv(r"c:\Users\Andi\learn_projects\GIT\UsedCarPrice\auto24\auto24_topredict_ebay.csv")

numeric_features = ['power_ps', 'mileage_km', 'car_age']
categorical_features = ['body_type', 'transmission', 'model', 'fuel_type', 'make', 'damaged', ]
#data[categorical_features] = data[categorical_features].fillna("unknown")

X_predict = auto24[categorical_features + numeric_features].copy()

In [27]:
#Standard HistGradientBoostRegressor
import time
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error, mean_absolute_percentage_error
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import ParameterGrid

ohe = OrdinalEncoder(handle_unknown="use_encoded_value",
    unknown_value=np.nan,
    encoded_missing_value=np.nan,
    max_categories=254,)
cat_indices = list(range(len(numeric_features), len(numeric_features) + len(categorical_features)))

seed = 42

def make_forest(loss = "absolute_error", learning_rate = 0.1, max_iter= 300, max_leaf_nodes = 31, min_samples_leaf = 5, l2_regularization=0.0,):
    prepare = ColumnTransformer([
        ("numeric", "passthrough", numeric_features),
        ("categorical", ohe, categorical_features),
    ])
    pipe = Pipeline([
        ("prepare", prepare),
        ("forest", HistGradientBoostingRegressor(loss=loss,
            learning_rate=learning_rate,
            max_iter=max_iter,
            max_leaf_nodes=max_leaf_nodes,
            min_samples_leaf=min_samples_leaf,
            l2_regularization=l2_regularization,
            categorical_features=cat_indices,
            early_stopping=True,      
            validation_fraction=0.1,
            n_iter_no_change=15,
            random_state=seed,)),
         ])
    return pipe
        
#Defining the hyperparameters and running the model:

def holdout_table(parameter_choices):
    rows, models = [], []
    for i, settings in enumerate(ParameterGrid(parameter_choices), start=1):
        t0 = time.time()
        model = make_forest(**settings).fit(X_train, y_train)
        models.append(model)
        probs_train = model.predict(X_train)
        probs_val = model.predict(X_val)
        
        predictors = model.named_steps["forest"]._predictors
        
        rows.append({
            "settings":settings,
            "training MAE": mean_absolute_error(y_train, probs_train), #which measure to use?!
            "validation MAE": mean_absolute_error(y_val, probs_val), #which measure to use?!
            "validation MAE in %": mean_absolute_percentage_error(y_val, probs_val)*100,
            "training RMSE": root_mean_squared_error(y_train, probs_train),
            "validation RMSE": root_mean_squared_error(y_val, probs_val),
            "val_r2": r2_score(y_val, probs_val),
            "avg_nodes": sum(p[0].nodes.shape[0] for p in predictors) / len(predictors),
        })
        print(f"[{i}/9] Finished {settings} in {time.time() - t0:.1f}s")
    return pd.DataFrame(rows), models

depth_results, depth_models = holdout_table({
    "loss" : ["absolute_error"], "learning_rate": [0.1], "max_iter" : [1000], "max_leaf_nodes" : [255], "min_samples_leaf" : [20,], "l2_regularization": [0.1],
})
leaf_labels = [str(settings["max_leaf_nodes"]) for settings in depth_results["settings"]]

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(depth_results.round(3))




[1/9] Finished {'l2_regularization': 0.1, 'learning_rate': 0.1, 'loss': 'absolute_error', 'max_iter': 1000, 'max_leaf_nodes': 255, 'min_samples_leaf': 20} in 57.3s


,settings,training MAE,validation MAE,validation MAE in %,training RMSE,validation RMSE,val_r2,avg_nodes
0,"{'l2_regularization': 0.1, 'learning_rate': 0....",0.191,0.214,2.717,0.281,0.307,0.91,509.0


In [31]:
best_idx = depth_results["validation MAE"].idxmin()
best_model = depth_models[best_idx]


probs_auto24 = best_model.predict(X_predict)
auto24["ebay_logprice_predicted"] = probs_auto24
auto24.to_csv(r"c:\Users\Andi\learn_projects\GIT\UsedCarPrice\auto24\auto24_ebay_predicted.csv")
print(probs_auto24)

[10.12392228  9.03485105 10.11458196 ...  9.06149673 10.12371204
  8.99987888]


In [ ]:
auto24 = pd.read_csv("C:\Users\Andi\learn_projects\GIT\UsedCarPrice\auto24\auto24_topredict_ebay.csv")

numeric_features = ['power_ps', 'mileage_km', 'car_age']
categorical_features = ['body_type', 'transmission', 'model', 'fuel_type', 'make', 'damaged', ]
#data[categorical_features] = data[categorical_features].fillna("unknown")

X_predict = data[categorical_features + numeric_features].copy()